# 03 — Clustering experiments

Sweep HDBSCAN hyperparameters on the bundled sample and pick the best (mean CV ARI, subject to noise fraction below 80%).

Compare HDBSCAN to a KMeans baseline as a sanity check.

In [ ]:
from pathlib import Path

import hdbscan
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data/raw/data_sensors.csv")
X = StandardScaler().fit_transform(df.drop(columns=["Label"]))
y = df["Label"].to_numpy()
labeled_mask = ~pd.isna(y)
print(f"X: {X.shape}, labeled: {labeled_mask.sum()}")

In [ ]:
# HDBSCAN sweep
rows = []
for mcs in [3, 5, 8, 10, 15, 20, 30]:
    for ms in [1, 3, 5]:
        m = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit(X)
        labels = m.labels_
        n_clusters = len(set(labels.tolist()) - {-1})
        noise = (labels == -1).mean()
        if (labels[labeled_mask] != -1).any():
            ari = adjusted_rand_score(y[labeled_mask], labels[labeled_mask])
        else:
            ari = float("nan")
        rows.append(
            {"mcs": mcs, "ms": ms, "n_clusters": n_clusters, "noise": noise, "ari_labeled": ari}
        )
sweep = pd.DataFrame(rows).sort_values("ari_labeled", ascending=False, na_position="last")
sweep

In [ ]:
# KMeans baseline
rows = []
for k in [2, 3, 4, 5, 8, 10]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    labels = km.labels_
    sil = silhouette_score(X, labels)
    ari = adjusted_rand_score(y[labeled_mask], labels[labeled_mask])
    rows.append({"k": k, "silhouette": sil, "ari_labeled": ari})
pd.DataFrame(rows)